# QUICK-COMMERCE DELIVERY INTELLIGENCE
## Exploratory Data Analysis, Spatial Auditing & Feature Dynamics

This notebook conducts a deep inspection and exploratory audit of the Quick-Commerce delivery dataset, verifies the data cleaning pipeline, calculates geospatial Haversine metrics, and establishes baseline distributions for ETA prediction and late-order breach detection.

In [ ]:
import sys
from pathlib import Path

# Add repository root to path for modular src imports
ROOT_DIR = Path.cwd().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import config
from src.data_loader import DataLoader
from src.data_cleaning import DataCleaner
from src.feature_engineering import FeatureEngineer
from src.eda import ExploratoryDataAnalysis

print("Project Root:", ROOT_DIR)
print("Raw Data Path:", config.RAW_DATA_PATH)

## 1. Data Ingestion & Structural Inspection

In [ ]:
loader = DataLoader()
df_raw = loader.load_raw_data()

is_valid, missing_cols = loader.validate_raw_schema(df_raw)
print(f"Schema Valid: {is_valid} | Missing Columns: {missing_cols}")

summary = loader.inspect_dataset(df_raw)
display(df_raw.head())

## 2. Data Cleaning & Sanitization
- Stripping trailing whitespace across categorical variables.
- Normalizing `Weatherconditions` (removing `conditions ` prefix).
- Extracting numeric minutes from target string `(min) 24` -> `24.0`.
- Rectifying sign flips in GPS coordinates and masking (0,0) invalid coordinates.
- Parsing dates and times into normalized minute-of-day offsets.

In [ ]:
cleaner = DataCleaner()
df_cleaned = cleaner.clean(df_raw, is_training=True)

print("Cleaned Data Dimensions:", df_cleaned.shape)
display(df_cleaned.head())

## 3. Feature Engineering & Geospatial Metrics
- **Haversine Geodesic Distance**: Great-circle distance between merchant restaurant and customer drop point.
- **Kitchen Preparation Latency**: Time difference between ordering and rider pickup.
- **Temporal Indicators**: Hour of order, day of week, rush hour flags.
- **SLA Breach Target**: Binary flag `is_late` based on delivery SLA cutoff (default 30 mins).

In [ ]:
engineer = FeatureEngineer(sla_threshold_min=config.DEFAULT_SLA_MINUTES)
df_featured = engineer.transform_features(df_cleaned, is_training=True)

print("Engineered Data Shape:", df_featured.shape)
print("Engineered Columns:", list(df_featured.columns))
display(df_featured[["distance_km", "prep_time_min", "order_hour", "traffic_density_ordinal", "delivery_time_min", "is_late"]].describe())

## 4. Visual Exploratory Analysis

In [ ]:
eda = ExploratoryDataAnalysis()
eda_figures = eda.run_full_eda(df_featured)
print("Saved EDA Visualizations:", list(eda_figures.keys()))

## 5. Next Steps for Modeling & Optimization
1. **Model Training**: Execute regression pipelines (Ridge, Random Forest, XGBoost) to predict `delivery_time_min`.
2. **Breach Classification**: Train imbalance-aware classifiers to predict `is_late` risk probabilities.
3. **Explainability**: Generate SHAP feature attributions.
4. **Spatial Optimization**: Formulate dark store cluster centroids using `DeliveryOptimizerClustering`.
5. **Deployment**: Integrate trained serialized artifacts into Streamlit dashboard.